# Код написан с помощью ChatGPT 6 Astra (Режим высокий-максимальный)

## *Данная тетрадка необязательна (простой пример запуска и проверка подготовки 5 мутаций).

# Проверка исходной ThermoMPNN

Получаем прогнозы для пяти одиночных мутаций 1A32 из train.
Это пример запуска; сравнение моделей проводится во второй тетрадке.

Нужны установленная ThermoMPNN и данные MegaScale в `data/megascale/`:
распакованные CSV/PDB либо архивы `Processed_K50_dG_datasets.zip`
и `AlphaFold_model_PDBs.zip`. Скачивание здесь не выполняется.

In [1]:
from pathlib import Path
import pickle
import sys
import zipfile

import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT
THERMOMPNN_DIR = PROJECT_ROOT / "external" / "ThermoMPNN"
DATA_DIR = PROJECT_ROOT / "data" / "megascale"
RESULTS_DIR = PROJECT_ROOT / "results"
DEVICE = torch.device("cpu")

protein_id = "1A32.pdb"
csv_path = (
    DATA_DIR / "Processed_K50_dG_datasets"
    / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv"
)
pdb_path = DATA_DIR / "AlphaFold_model_PDBs" / protein_id

## Модель

Загружаем готовые веса и включаем режим предсказания на CPU.
Обучение в этой тетрадке не проводится.

In [2]:
if str(THERMOMPNN_DIR) not in sys.path:
    sys.path.insert(0, str(THERMOMPNN_DIR))

from train_thermompnn import TransferModelPL
from protein_mpnn_utils import parse_PDB
from datasets import Mutation

cfg = OmegaConf.load(THERMOMPNN_DIR / "config.yaml")
cfg.platform = {"thermompnn_dir": str(THERMOMPNN_DIR), "accel": "cpu"}

model = TransferModelPL.load_from_checkpoint(
    str(THERMOMPNN_DIR / "models" / "thermoMPNN_default.pt"),
    cfg=cfg, map_location=DEVICE, strict=True,
).eval()

/home/andre/miniconda3/envs/NeuroPP/lib/python3.10/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
/home/andre/miniconda3/envs/NeuroPP/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary obj

## Данные

Проверяем, что 1A32 входит в официальный train, и при необходимости
извлекаем CSV и PDB из уже скачанных архивов.
Из таблицы читаем только записи выбранного белка.

In [3]:
with (THERMOMPNN_DIR / "dataset_splits" / "mega_splits.pkl").open("rb") as file:
    splits = pickle.load(file)
assert protein_id in splits["train"], "Белок должен входить в train."

for path, archive_name in [
    (csv_path, "Processed_K50_dG_datasets.zip"),
    (pdb_path, "AlphaFold_model_PDBs.zip"),
]:
    if not path.exists():
        with zipfile.ZipFile(DATA_DIR / archive_name) as archive:
            archive.extract(path.relative_to(DATA_DIR).as_posix(), DATA_DIR)

In [4]:
with pd.read_csv(
    csv_path,
    usecols=["WT_name", "mut_type", "aa_seq", "ddG_ML"],
    dtype=str, keep_default_na=False, chunksize=50_000,
) as reader:
    protein_data = pd.concat(
        (chunk.loc[chunk["WT_name"] == protein_id] for chunk in reader),
        ignore_index=True,
    )

protein_data["ddG_ML"] = pd.to_numeric(protein_data["ddG_ML"], errors="coerce")

## Соответствие структуры и последовательности

Ожидаем одну цепь, совпадение последовательности WT с PDB
и полные координаты атомов, используемых моделью.

In [5]:
pdb_batch = parse_PDB(str(pdb_path))
assert len(pdb_batch) == 1, "Ожидалась одна структура."
pdb_record = pdb_batch[0]
assert pdb_record["num_of_chains"] == 1, "Ожидалась одна цепь."

wt_sequences = protein_data.loc[
    protein_data["mut_type"] == "wt", "aa_seq"
].drop_duplicates()
assert len(wt_sequences) == 1, "Ожидалась одна последовательность WT."
wt_sequence = wt_sequences.iloc[0]
assert wt_sequence == pdb_record["seq"], "Последовательности WT и PDB различаются."

coords_key = next(key for key in pdb_record if key.startswith("coords_chain_"))
assert all(
    np.isfinite(xyz).all() for xyz in pdb_record[coords_key].values()
), "В структуре есть пропуски координат."

## Мутации и знак эффекта

Оставляем одиночные замены с конечной числовой меткой.
Номер позиции начинается с 1, индекс для модели — с 0.
Проверяем WT-букву и последовательность каждого мутанта.

**`ddg_exp = -ddG_ML`: положительное значение означает дестабилизацию.**
Единицы эффекта и ошибки — ккал/моль.

In [6]:
amino_acids = "ACDEFGHIKLMNPQRSTVWY"
is_single = protein_data["mut_type"].str.fullmatch(
    rf"[{amino_acids}][1-9][0-9]*[{amino_acids}]", na=False
)
single_mutants = protein_data.loc[
    is_single & np.isfinite(protein_data["ddG_ML"])
].copy()

single_mutants["wt_aa"] = single_mutants["mut_type"].str[0]
single_mutants["mut_aa"] = single_mutants["mut_type"].str[-1]
single_mutants["position"] = single_mutants["mut_type"].str[1:-1].astype(int)
single_mutants["index0"] = single_mutants["position"] - 1
single_mutants = single_mutants.loc[
    single_mutants["wt_aa"] != single_mutants["mut_aa"]
].copy()
single_mutants["ddg_exp"] = -single_mutants["ddG_ML"]

for row in single_mutants.itertuples():
    i = row.index0
    assert 0 <= i < len(wt_sequence), f"Позиция вне белка: {row.mut_type}"
    assert wt_sequence[i] == row.wt_aa, f"Неверная WT-буква: {row.mut_type}"
    assert row.aa_seq == wt_sequence[:i] + row.mut_aa + wt_sequence[i + 1:], (
        f"Неверная последовательность мутанта: {row.mut_type}"
    )

examples = single_mutants.sample(n=5, random_state=42)

## Прогнозы и ошибки

Передаём модели структуру WT и пять мутаций.
Экспериментальные метки используются только для расчёта ошибок.
Сохраняем итоговую таблицу в `results/1A32_first_predictions.csv`.

In [7]:
mutations = [
    Mutation(int(row.index0), row.wt_aa, row.mut_aa)
    for row in examples.itertuples()
]
with torch.inference_mode():
    predictions, _ = model(pdb_batch, mutations)
assert len(predictions) == len(examples), "Число прогнозов не совпало с числом мутаций."

In [8]:
results = examples[
    ["mut_type", "position", "index0", "ddg_exp"]
].reset_index(drop=True)
results["ddg_pred"] = [prediction["ddG"].item() for prediction in predictions]
assert np.isfinite(results["ddg_pred"]).all(), "Получены некорректные прогнозы."
results["error"] = results["ddg_pred"] - results["ddg_exp"]
results["abs_error"] = results["error"].abs()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(RESULTS_DIR / f"{Path(protein_id).stem}_first_predictions.csv", index=False)

display(results.round(4))
print(f"MAE на пяти примерах: {results['abs_error'].mean():.4f} ккал/моль")

,mut_type,position,index0,ddg_exp,ddg_pred,error,abs_error
0,L34A,34,33,-0.0507,-0.0955,-0.0448,0.0448
1,I13Y,13,12,1.4677,1.0422,-0.4255,0.4255
2,I58T,58,57,0.6757,0.8818,0.2061,0.2061
3,V59L,59,58,-0.2619,0.1746,0.4365,0.4365
4,V4E,4,3,-0.3601,-0.2813,0.0788,0.0788


MAE на пяти примерах: 0.2383 ккал/моль


Пять мутаций белка из train проверяют работу модели и подготовку входов.
Их MAE не оценивает качество на новых белках.
Для этого используется полный эксперимент во второй тетрадке.